# Descarga de datos – API-Football (plan gratuito) – todas las categorías del fútbol argentino
**Cómo usarlo:**
1. En Colab, barra izquierda → 🔑 *Secretos* → agregar `API_FOOTBALL_KEY` con tu clave y activar el acceso.
2. Menú **Entorno de ejecución → Ejecutar todo**.
3. Aceptá el permiso de Google Drive cuando lo pida.

⚠️ Con todas las categorías son ≈ 150-200 consultas: **no entra en un día** (cupo: 100).
El notebook se detiene solo cuando se acaba el cupo. **Al día siguiente (después de las 21:00 hs de Argentina) volvé a ejecutar todo**: retoma donde quedó sin repetir consultas.
Orden: Primera → Primera Nacional → B Metro → Federal A → Primera C → Primera D → resto.

In [ ]:
from google.colab import drive, userdata   # Drive para guardar; userdata para leer la clave secreta
import requests, json, os, time, glob       # consultas HTTP, archivos JSON, pausas, búsqueda de archivos
import pandas as pd                         # tablas

drive.mount('/content/drive')               # conecta Google Drive
RAIZ = '/content/drive/MyDrive/taller-futbol/data'       # carpeta del proyecto en Drive
CARPETA = f'{RAIZ}/raw/api_football'                     # respuestas crudas
PROCESADOS = f'{RAIZ}/processed'                         # tablas limpias
os.makedirs(CARPETA, exist_ok=True); os.makedirs(PROCESADOS, exist_ok=True)

BASE = 'https://v3.football.api-sports.io'               # dirección de la API
HEADERS = {'x-apisports-key': userdata.get('API_FOOTBALL_KEY').strip()}  # .strip() saca espacios y saltos de línea

In [ ]:
import re   # para leer años dentro de los mensajes de error

class SinCupo(Exception):
    """Se lanza solo cuando se acaba el cupo DIARIO: corta la ejecución limpiamente."""
    pass

TEMPORADAS_PERMITIDAS = None   # se completa sola con lo que diga la API

def consultar(endpoint, params, nombre, reintento=True):
    """Consulta una sola vez; si el JSON ya existe lo reutiliza (no gasta cupo)."""
    global TEMPORADAS_PERMITIDAS
    ruta = os.path.join(CARPETA, nombre)
    if os.path.exists(ruta):                               # ya descargado: no gasta cupo
        return json.load(open(ruta))
    r = requests.get(f'{BASE}/{endpoint}', headers=HEADERS, params=params)
    time.sleep(7)                                          # pausa SIEMPRE, también si hay error
    data = r.json()
    errores = data.get('errors')
    if errores:
        texto = str(errores)
        if 'rateLimit' in texto and reintento:             # límite POR MINUTO: esperamos y reintentamos
            print('⏳ Límite por minuto, espero 65 s...'); time.sleep(65)
            return consultar(endpoint, params, nombre, reintento=False)
        if 'day' in texto.lower() or 'request limit' in texto.lower():   # cupo DIARIO agotado
            raise SinCupo('Se acabó el cupo de hoy. Volvé a ejecutar todo después de las 21:00 (hora Argentina).')
        if 'plan' in errores and TEMPORADAS_PERMITIDAS is None:          # temporada no permitida en plan gratis
            anios = [int(a) for a in re.findall(r'(20\d\d)', texto)]
            if len(anios) >= 2:
                TEMPORADAS_PERMITIDAS = list(range(min(anios), max(anios) + 1))
                print('ℹ️ El plan gratis permite las temporadas:', TEMPORADAS_PERMITIDAS)
        print('⚠️', nombre, errores)
        return None
    json.dump(data, open(ruta, 'w'))                       # guardamos crudo
    restantes = r.headers.get('x-ratelimit-requests-remaining')
    print(f"✔ {nombre} | quedan hoy: {restantes}")
    if restantes is not None and int(restantes) <= 0:
        raise SinCupo('Cupo agotado. Volvé a ejecutar todo después de las 21:00 (hora Argentina).')
    return data

estado = requests.get(f'{BASE}/status', headers=HEADERS).json()   # verifica la clave
print('Errores:', estado.get('errors'))
print('Cuenta:', estado.get('response'))

In [ ]:
# 1) Todas las ligas de Argentina (1 consulta)
ligas = consultar('leagues', {'country': 'Argentina'}, 'leagues_argentina.json')

PRIORIDAD = ['liga profesional', 'primera división', 'primera nacional', 'primera b metropolitana', 'b metro',
             'federal a', 'primera c', 'primera d']           # orden de importancia

def prioridad(nombre):
    n = nombre.lower()
    for i, k in enumerate(PRIORIDAD):
        if k in n:
            return i
    return len(PRIORIDAD)                                  # el resto, al final

# Solo ligas (no copas: sus equipos ya están en alguna liga)
LIGAS = sorted([(i['league']['id'], i['league']['name'], i['seasons'])
                for i in ligas['response'] if i['league']['type'] == 'League'],
               key=lambda x: prioridad(x[1]))

print('Ligas que se van a descargar, en orden:')
for lid, nombre, seasons in LIGAS:
    anios = [s['year'] for s in seasons]
    print(f'  {lid:>5}  {nombre:<45} temporadas: {min(anios)}–{max(anios)}')

In [ ]:
# 2) Equipos de cada liga en la temporada más reciente permitida (1-3 consultas por liga)
def equipos_de(liga, seasons):
    disponibles = sorted([s['year'] for s in seasons], reverse=True)          # años que tiene la liga
    for anio in disponibles:
        if TEMPORADAS_PERMITIDAS and anio not in TEMPORADAS_PERMITIDAS:        # saltear años no permitidos (no gasta cupo)
            continue
        eq = consultar('teams', {'league': liga, 'season': anio}, f'teams_{liga}_{anio}.json')
        if eq and eq['response']:
            return [e['team']['id'] for e in eq['response']], anio
    return [], None

ids_equipos, resumen = [], []
try:
    for lid, nombre, seasons in LIGAS:
        ids, anio = equipos_de(lid, seasons)
        resumen.append({'liga_id': lid, 'liga': nombre, 'temporada': anio, 'equipos': len(ids)})
        ids_equipos += [i for i in ids if i not in ids_equipos]   # une sin repetir
except SinCupo as e:
    print('⛔', e)
display(pd.DataFrame(resumen))
print('Total de clubes a consultar:', len(ids_equipos))

In [ ]:
# 3) Transferencias de cada club: toda su historia (1 consulta por club)
hechos = 0
try:
    for team_id in ids_equipos:
        consultar('transfers', {'team': team_id}, f'transfers_team_{team_id}.json')
        hechos += 1
    print(f'✅ Transferencias completas: {hechos}/{len(ids_equipos)} clubes')
except SinCupo as e:
    print(f'⛔ {e}  Avance: {hechos}/{len(ids_equipos)} clubes')

In [ ]:
# 3b) Lista de TODOS los clubes argentinos (1 consulta) para saber si un destino es del país o del exterior
try:
    arg = consultar('teams', {'country': 'Argentina'}, 'teams_country_argentina.json')
    if arg:
        pd.DataFrame([{'team_id': e['team']['id'], 'nombre': e['team']['name'], 'ciudad': e['venue']['city']}
                      for e in arg['response']]).to_csv(f'{PROCESADOS}/apifootball_clubes_argentinos.csv', index=False)
        print('Clubes argentinos:', len(arg['response']))
except SinCupo as e:
    print('⛔', e)

In [ ]:
# 4) Pasar transferencias a CSV
filas = []
for ruta in glob.glob(f'{CARPETA}/transfers_team_*.json'):
    for item in json.load(open(ruta))['response']:
        for t in item['transfers']:
            filas.append({'player_id': item['player']['id'], 'player_name': item['player']['name'],
                          'fecha': t['date'], 'tipo': t['type'],
                          'club_salida_id': t['teams']['out']['id'], 'club_salida': t['teams']['out']['name'],
                          'club_llegada_id': t['teams']['in']['id'], 'club_llegada': t['teams']['in']['name']})
df = pd.DataFrame(filas).drop_duplicates(subset=['player_id', 'fecha', 'club_salida_id', 'club_llegada_id'])
df.to_csv(f'{PROCESADOS}/apifootball_transfers.csv', index=False)
print(df.shape, '| clubes con datos:', len(glob.glob(f'{CARPETA}/transfers_team_*.json')))
df.head()

## (Opcional, otro día) Jugadores y minutos por temporada
Gasta 30-45 consultas por temporada. Descomentá la última línea y ejecutá **un día distinto** al de las transferencias.

In [ ]:
def bajar_temporada(liga, anio):
    pagina = 1
    while True:
        data = consultar('players', {'league': liga, 'season': anio, 'page': pagina}, f'players_{liga}_{anio}_p{pagina}.json')
        if data is None or pagina >= data['paging']['total']:
            break
        pagina += 1

# Ejemplo: jugadores de Primera en la temporada usada (30-45 consultas). Ejecutar otro día.
# bajar_temporada(LIGAS[0][0], resumen[0]['temporada'])